# 05. Full-Prompt (v0_6) NL-to-JOILang + IR Evaluation on JOICommands-280

GPS 논문의 baseline 조건(full prompt)을 재현합니다: `ModelMagementApp/gpt_mg/version0_6`의
monolith 프롬프트를 그대로 쓰되 **service list만 ver2.0.1로 교체**하고, JOICommands-280 전량에 대해
cloud(gpt-4.1-mini = CloudRef)와 local(Qwen2.5-Coder-7B = DirectTransfer)을 생성·평가합니다.
평가는 `JOILang-Server/utils/evaluation`(SOPLang IR 파서 + z3 동치)와 strict DET
(`utils/det_evaluator`)를 함께 사용합니다.

```text
파이프라인 게이트 & 단계
 1. 환경변수 확인            : OPENAI_API_KEY / JOI_EVAL_* / HF 캐시
 2. Premapping recall 게이트 : 280 전행에서 gt 필요 장치 ⊆ shortlist (요구: retrieval 행 140/140)
 3. Full prompt 조립         : v0_6 md 자산(읽기 전용) + ver2.0.1 service list(호출명 규약 변환)
 4. Cloud 생성               : gpt-4.1-mini, temperature 0, 재개 가능 루프
 5. Local 생성               : Qwen2.5-Coder-7B-Instruct (local_models, bf16, greedy)
 6. 평가                     : utils/evaluation.compare_codes (logic_equiv / script_sim / ast_sim)
                              + strict DET (det_pass@70) -> 논문 표와 비교
```

논문 목표치 (Table 5; 사전 측정 전 값이므로 "유사 수준" 확인이 목적):

| 조건 | DETPass(%) | S_DET | 입력 토큰 | 지연(s) |
|---|---|---|---|---|
| GPT-4.1-mini CloudRef | 94.5 | 0.89 | 3,500 | 2.85 |
| Qwen2.5-Coder-7B DirectTransfer | 87.8 | 0.73 | 3,500 | 2.28 |

주의:

- `ModelMagementApp`은 읽기 전용입니다. v0_6의 `config_loader.py`는 실행 시 자기 폴더에
  merged prompt 파일을 쓰므로 직접 실행하지 않고, 동일한 조립 로직을 이 노트북 안에서 재현합니다
  (병합 프롬프트는 artifacts에 저장).
- v0_6 프롬프트 계약은 "[Service List]에 정의된 이름 그대로 호출"이며, 구 목록은 service 필드가
  곧 호출명이었습니다. 따라서 ver2.0.1 목록도 service 필드를 런타임 호출명
  `{device}_{service}`(소문자, 280 gt 규약)으로 변환해 주입합니다 — 목록만 교체한다는 원칙의
  기능적 해석입니다.
- 결정성을 위해 temperature 0(greedy)로 고정합니다(원본 model_config에는 temperature 미지정 = API 기본 1.0).
- ReplayFit 열은 리플레이 셋이 아직 없어 비교에서 제외합니다.
- API 키는 셸 환경변수에서만 읽습니다. 노트북/저장소에 키를 쓰지 않습니다.


## 1. Setup & 환경변수 확인

모든 노브는 환경변수로 재정의 가능합니다. 생성 루프는 jsonl에 증분 저장되며 재실행 시 이어서 돕니다.

In [1]:
# ============================================================
# Setup
# ============================================================
import ast as pyast
import csv, json, os, re, sys, time
from pathlib import Path

import numpy as np

BASE_DIR = Path(os.environ.get("JOILANG_BASE_DIR", "/home/mgjeong/Desktop/llm/JOILang-Server"))
MMA_DIR = Path(os.environ.get("MODELMAGEMENT_DIR", "/home/mgjeong/Desktop/llm/ModelMagementApp"))  # read-only
V06_DIR = MMA_DIR / "gpt_mg" / "version0_6"
V15_DIR = BASE_DIR / "gpt_mg" / "version0_15_update20260413"
OPEN_DATA = Path(os.environ.get("JOILANG_OPEN_DATA", "/home/mgjeong/Desktop/llm/joilang_open_release/data"))
PREMAP_ART = BASE_DIR / "artifacts" / "premapping_pipeline_runs" / os.environ.get("PREMAP_RUN_TAG", "v201_paper")

RUN_TAG = os.environ.get("RUN_TAG", "fullprompt_v06_ver201")
ART_DIR = BASE_DIR / "artifacts" / "fullprompt_runs" / RUN_TAG
ART_DIR.mkdir(parents=True, exist_ok=True)

N_ROWS = int(os.environ.get("N_ROWS", "280"))
RUN_CLOUD = os.environ.get("RUN_CLOUD", "true").lower() == "true"
RUN_LOCAL = os.environ.get("RUN_LOCAL", "true").lower() == "true"
CLOUD_MODEL = os.environ.get("CLOUD_MODEL", "gpt-4.1-mini")
LOCAL_MODEL_DIR = os.environ.get("LOCAL_MODEL_DIR", "/home/mgjeong/Desktop/llm/local_models/qwen25_coder_7b")
FIXED_NOW = os.environ.get("FIXED_NOW", "2026-07-31 09:00:00")   # deterministic "Current Time"
MAX_NEW_TOKENS = int(os.environ.get("MAX_NEW_TOKENS", "1024"))
DET_THRESHOLD = float(os.environ.get("DET_THRESHOLD", "70"))

for p in (BASE_DIR, MMA_DIR, V06_DIR, V15_DIR):
    assert p.exists(), f"missing: {p}"

# --- environment variable check (presence only; values never printed) ---
def has(name):
    return "set" if os.environ.get(name, "").strip() else "MISSING"
for var in ("OPENAI_API_KEY", "OPENAI_BASE_URL", "JOI_EVAL_OPENAI_API_KEY",
            "JOI_V15_OPENAI_API_KEY", "HF_HOME", "HF_HUB_CACHE"):
    print(f"{var:26s}: {has(var)}")
assert os.environ.get("OPENAI_API_KEY", "").strip(), "export OPENAI_API_KEY before starting the kernel"
print("LangSmith tracing:", os.environ.get("LANGSMITH_TRACING", "unset"), "(off expected)")

with (BASE_DIR / "datasets" / "JOICommands-280.csv").open(encoding="utf-8-sig") as f:
    rows280 = list(csv.DictReader(f))
print(f"\nrows: {len(rows280)} | evaluating first N_ROWS = {N_ROWS} | artifacts: {ART_DIR}")

OPENAI_API_KEY            : set
OPENAI_BASE_URL           : set
JOI_EVAL_OPENAI_API_KEY   : set
JOI_V15_OPENAI_API_KEY    : set
HF_HOME                   : set
HF_HUB_CACHE              : set
LangSmith tracing: false (off expected)

rows: 280 | evaluating first N_ROWS = 280 | artifacts: /home/mgjeong/Desktop/llm/JOILang-Server/artifacts/fullprompt_runs/fullprompt_v06_ver201


## 2. Premapping recall 게이트 (파이프라인 진행 조건)

fine-tuned e5 번들 + mandatory 카테고리 보강(시간→Clock, on-off/지속시간→Switch, 먼지→실내외 겸비)
+ shortlist k=12(논문 운영점)로, gt가 요구하는 장치가 shortlist에 **모두** 포함되는지 280 전행에서
확인합니다. 요구 장치는 gt 스크립트의 `{device}_{service}` 호출 접두로 추출합니다.

- retrieval 행(연결 장치 없음, 140행): shortlist가 요구 장치를 전부 덮어야 함 -> **요구: 140/140**
- connected-devices 행(140행): retrieval이 생략되고 연결 장치가 문맥이 됨. gt가 연결 카테고리 밖
  장치(Clock 등 주변 장치, 태그 결합 서비스)를 쓰는 행이 일부 있으나, full-prompt 조건에서는
  전체 카탈로그가 프롬프트에 있어 영향이 없습니다(참고용으로 함께 보고).

In [2]:
# ============================================================
# Premap recall gate over all 280 rows
# ============================================================
os.environ.setdefault("JOI_V15_RETRIEVAL_PYTHON", sys.executable)
sys.path.insert(0, str(V15_DIR))
from utils.pipeline_common import SERVICE_SCHEMA_DEFAULT, load_service_schema, \
    build_service_snippet_payload, parse_connected_devices
from utils.retrieval_context import load_retrieval_config

schema = load_service_schema(SERVICE_SCHEMA_DEFAULT)
DEV_BY_LOWER = {d.lower(): d for d in schema}
gate_cfg = load_retrieval_config({
    "service_context_mode": "retrieval_fallback", "retrieval_topk": 12, "retrieval_mode": "hybrid",
    "retrieval_json_path": str(MMA_DIR / "datasets" / "service_list_ver2.0.1_retrieval_eng.json"),
    "retrieval_bundle_dir": str(PREMAP_ART / "retrieval_bundle_e5_ft"),
    "retrieval_model_dir": str(PREMAP_ART / "models" / "e5-small-split-v201"),
    "retrieval_device": "cpu",
})
CALL_RE = re.compile(r"\)\s*\.\s*([a-z][a-z0-9_]*)")

def required_devices(gt_script):
    req = set()
    for name in CALL_RE.findall(gt_script.lower()):
        dev = name.split("_", 1)[0]
        if dev in DEV_BY_LOWER:
            req.add(DEV_BY_LOWER[dev])
    return req

ret_full = ret_miss = cd_full = cd_miss = n_req = n_hit = 0
miss_detail = []
t0 = time.perf_counter()
for i, r in enumerate(rows280):
    req = required_devices(json.loads(r["gt"]).get("script", ""))
    if str(r.get("connected_devices", "")).strip():
        cd = parse_connected_devices(r["connected_devices"])
        cats = set()
        for info in cd.values():
            c = info.get("category")
            for x in (c if isinstance(c, list) else [c]):
                if x:
                    cats.add(str(x))
        if req - cats:
            cd_miss += 1
        else:
            cd_full += 1
        continue
    payload, _ = build_service_snippet_payload(r["command_eng"], {}, schema, retrieval_config=gate_cfg)
    short = {h["category"] for h in payload["retrieval"].get("shortlist", [])}
    n_req += len(req); n_hit += len(req & short)
    if req - short:
        ret_miss += 1
        miss_detail.append((i + 1, sorted(req - short), r["command_eng"][:60]))
    else:
        ret_full += 1

print(f"retrieval rows   : full-coverage {ret_full}/{ret_full + ret_miss} "
      f"| device recall {n_hit}/{n_req} = {n_hit / max(1, n_req):.4f}")
print(f"connected rows   : gt devices within connected categories {cd_full}/{cd_full + cd_miss} "
      f"(참고: full-prompt에서는 전체 카탈로그가 포함되어 영향 없음)")
print(f"gate time: {time.perf_counter() - t0:.0f}s")
for m in miss_detail:
    print("  MISS", m)
assert ret_miss == 0, "premapping recall gate failed - do not proceed to generation"
print("\nGATE PASSED: premapping device recall = 1.0 on all retrieval rows")

# free the "utils" package name (version0_15) for the server-side utils.evaluation import later
sys.path.remove(str(V15_DIR))
for k in list(sys.modules):
    if k == "utils" or k.startswith("utils."):
        del sys.modules[k]

retrieval rows   : full-coverage 140/140 | device recall 263/263 = 1.0000
connected rows   : gt devices within connected categories 127/140 (참고: full-prompt에서는 전체 카탈로그가 포함되어 영향 없음)
gate time: 34s

GATE PASSED: premapping device recall = 1.0 on all retrieval rows


## 3. Full prompt 조립 (v0_6 자산 + ver2.0.1 service list)

`config_loader.py`의 시스템 프롬프트 템플릿을 그대로 재현하되:

- md 자산 5종(`grammar_ver1.5.10`, `service_prompt_10`, `tempo_prompt_9`, `caution_prompt_8`,
  `response_prompt_6`)은 v0_6에서 읽기 전용으로 로드
- service list는 ver2.0.1 평탄화 목록에서 `service` 필드를 런타임 호출명
  `{device}_{service}`(소문자)으로 바꿔 주입
- `connected_devices`는 원본과 같은 `[#Tag, ...]` 블록(카테고리가 리스트인 신규 스키마 대응)
- 사용자 메시지는 `run.py`와 동일: `Current Time: ...\n\nGenerate JOI Lang code for Natural Language: ...`

병합 프롬프트 스냅샷은 artifacts에 저장하고, 입력 토큰 수를 Qwen 토크나이저로 측정합니다.

In [3]:
# ============================================================
# Assemble the v0_6 monolithic system prompt with ver2.0.1 lists
# ============================================================
def read_v06(name):
    return (V06_DIR / name).read_text(encoding="utf-8")

grammar = read_v06("grammar_ver1.5.10.md")
service_prompt = read_v06("service_prompt_10.md")
tempo = read_v06("tempo_prompt_9.md")
caution = read_v06("caution_prompt_8.md")
responsestep = read_v06("response_prompt_6.md")

def to_call_name(entry):
    return f"{entry['device']}_{entry['service']}".lower()

def load_call_list(path):
    out = []
    for e in json.loads(Path(path).read_text(encoding="utf-8")):
        item = dict(e)
        item["service"] = to_call_name(e)
        out.append(item)
    return out

service_list_value = load_call_list(OPEN_DATA / "service_list_ver2.0.1_value.json")
service_list_function = load_call_list(OPEN_DATA / "service_list_ver2.0.1_function.json")
print(f"service lists: {len(service_list_value)} VALUE / {len(service_list_function)} FUNCTION "
      f"(service field -> runtime call name, e.g. {service_list_function[0]['service']!r})")

def connected_block(connected_devices_str):
    s = str(connected_devices_str or "").strip()
    if not s:
        return ""
    try:
        cd = pyast.literal_eval(s)
    except Exception:
        return ""
    cats = set()
    for info in cd.values():
        c = info.get("category")
        for x in (c if isinstance(c, list) else [c]):
            if x:
                cats.add(str(x))
    if not cats:
        return ""
    tags = "[" + ", ".join(f"#{c}" for c in sorted(cats)) + "]"
    return f"\n\n---\n[connected_devices]\n {tags}"

def build_system_prompt(connected_devices_str=""):
    connected_devices = connected_block(connected_devices_str)
    other_params = ""
    return f"""
You are a JOILang programmer. JOILang is a programming language used to control IoT devices.
Use the following knowledge to convert natural language into valid JOILang code.

Make sure to follow syntax rules strictly. Only use allowed keywords:
if, else if, else, >=, <=, ==, !=, not, and, or, wait until, (#Clock).clock_delay() 
The delay function (#Clock).clock_delay() only accepts values in milliseconds (ms).
Do not use while or any unlisted constructs. 
**Never use `while` in code**
[Incorrect Example]
while (blinkCount < 10)

---

[Device and Service Mapping]
IMPORTANT: You MUST extract **all device tags mentioned as subjects or objects in the input sentence**, including those connected by conjunctions such as "and" or "with".  
For each extracted device tag, retrieve **all associated services** (both value and function names) exactly as defined in the [Service List].  
**Do not omit any device or service even if their names overlap or repeat.**  
If multiple devices share similar service names (e.g., "alarm" function on both Alarm and Siren devices), include the services for each device separately and comprehensively.  
{service_prompt}
[service_list_value]
{service_list_value}
[service_list_function]
{service_list_function}

---
[Grammar]
{grammar}


---
[Condition Combination Rules]
{tempo}


---
[Important Cautions]
{caution}
{connected_devices}
{other_params}

---
{responsestep}
- **Never use `while` in code**

--- JOILang Code Output Format Guide ---
Every scenario generated will follow this structure:
```json
{{
  "name": "<명령의 의도를 한국어로 **축약하여**, 띄어쓰기 없이 간결한 형태로 작성하세요. 너무 길게 쓰지 말고, 조합된 단어로 의미만 담아내세요.>",
  "cron": "<Time-based trigger to start execution>",
  "period": <Execution interval in milliseconds or -1>,
  "code": "<Main logic block written in JOILang>"
}}
```

"""

def user_prompt(sentence):
    return f"Current Time: {FIXED_NOW}\n\nGenerate JOI Lang code for Natural Language: {sentence}"

base_system = build_system_prompt("")
(ART_DIR / "merged_system_prompt_ver201.md").write_text(base_system, encoding="utf-8")
print(f"system prompt: {len(base_system):,} chars -> {ART_DIR / 'merged_system_prompt_ver201.md'}")

from transformers import AutoTokenizer
qwen_tok = AutoTokenizer.from_pretrained(LOCAL_MODEL_DIR)
n_sys_tokens = len(qwen_tok(base_system)["input_ids"])
print(f"system prompt tokens (Qwen tokenizer): {n_sys_tokens:,} "
      f"(논문 표의 3,500은 사전 측정 전 값 - full prompt 실측은 이 값 기준)")
assert n_sys_tokens + MAX_NEW_TOKENS + 256 < 32768, "prompt would overflow Qwen 32k context"

service lists: 114 VALUE / 81 FUNCTION (service field -> runtime call name, e.g. 'airconditioner_setairconditionermode')
system prompt: 114,512 chars -> /home/mgjeong/Desktop/llm/JOILang-Server/artifacts/fullprompt_runs/fullprompt_v06_ver201/merged_system_prompt_ver201.md


/home/mgjeong/miniconda3/envs/joi/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


system prompt tokens (Qwen tokenizer): 30,035 (논문 표의 3,500은 사전 측정 전 값 - full prompt 실측은 이 값 기준)


## 4. Cloud 생성: gpt-4.1-mini (CloudRef)

행별로 system(연결 장치 블록 포함) + user 메시지를 만들어 호출합니다. temperature 0, 결과는
`cloud_generations.jsonl`에 증분 저장되어 재실행 시 이어서 생성합니다. usage의 prompt/completion
토큰과 지연을 함께 기록합니다.

In [4]:
# ============================================================
# Cloud generation (resume-capable)
# ============================================================
def parse_joilang_json(text):
    """Lenient JOILang-JSON extraction.

    Model outputs frequently contain RAW newlines inside the "code" string
    (invalid JSON); the original v0_6 run.py salvaged such outputs, so we do
    too: try strict json.loads first, then re-escape newlines inside the
    code/script string value, then fall back to field-level regex extraction.
    """
    t = re.sub(r"^```(?:json)?\s*|\s*```$", "", str(text).strip(), flags=re.S)
    m = re.search(r"\{.*\}", t, flags=re.S)
    if not m:
        return None
    raw = m.group(0)
    try:
        return json.loads(raw)
    except Exception:
        pass
    m2 = re.search(r'("(?:code|script)"\s*:\s*")(.*?)("\s*\}?\s*$)', raw, flags=re.S)
    if m2:
        fixed = raw[:m2.start(2)] + m2.group(2).replace("\\", "\\\\").replace("\n", "\\n").replace("\t", "\\t") + raw[m2.end(2):]
        try:
            return json.loads(fixed)
        except Exception:
            pass
    out = {}
    for key in ("name", "cron"):
        km = re.search(rf'"{key}"\s*:\s*"([^"]*)"', raw)
        out[key] = km.group(1) if km else ""
    pm = re.search(r'"period"\s*:\s*(-?\d+)', raw)
    out["period"] = int(pm.group(1)) if pm else -1
    cm = re.search(r'"(?:code|script)"\s*:\s*"(.*)"\s*\}?\s*$', raw, flags=re.S)
    if not cm:
        return None
    out["code"] = cm.group(1).replace('\\"', '"')
    return out

def load_done(path):
    done = {}
    if Path(path).exists():
        for line in Path(path).read_text(encoding="utf-8").splitlines():
            if line.strip():
                rec = json.loads(line)
                done[rec["row"]] = rec
    return done

CLOUD_OUT = ART_DIR / "cloud_generations.jsonl"
if RUN_CLOUD:
    from openai import OpenAI
    client = OpenAI()   # OPENAI_API_KEY / OPENAI_BASE_URL from the environment
    done = load_done(CLOUD_OUT)
    todo = [i for i in range(min(N_ROWS, len(rows280))) if i not in done]
    print(f"cloud: {len(done)} cached, {len(todo)} to generate")
    with CLOUD_OUT.open("a", encoding="utf-8") as f:
        for n, i in enumerate(todo, 1):
            r = rows280[i]
            msgs = [{"role": "system", "content": build_system_prompt(r.get("connected_devices", ""))},
                    {"role": "user", "content": user_prompt(r["command_eng"])}]
            t0 = time.perf_counter()
            try:
                resp = client.chat.completions.create(model=CLOUD_MODEL, messages=msgs,
                                                      temperature=0, max_tokens=MAX_NEW_TOKENS)
                content = resp.choices[0].message.content
                usage = {"prompt_tokens": resp.usage.prompt_tokens,
                         "completion_tokens": resp.usage.completion_tokens}
                err = ""
            except Exception as exc:
                content, usage, err = "", {}, str(exc)[:200]
            rec = {"row": i, "category": r["category"], "index": r["index"],
                   "command_eng": r["command_eng"], "content": content,
                   "parsed": parse_joilang_json(content), "usage": usage,
                   "latency_sec": round(time.perf_counter() - t0, 3), "error": err}
            f.write(json.dumps(rec, ensure_ascii=False) + "\n"); f.flush()
            if n % 20 == 0 or n == len(todo):
                print(f"  cloud {n}/{len(todo)} (row {i + 1}) {rec['latency_sec']}s", flush=True)
    cloud_recs = load_done(CLOUD_OUT)
    n_err = sum(1 for x in cloud_recs.values() if x["error"])
    print(f"cloud done: {len(cloud_recs)} rows, {n_err} API errors, "
          f"avg prompt tokens {np.mean([x['usage'].get('prompt_tokens', 0) for x in cloud_recs.values()]):.0f}, "
          f"avg latency {np.mean([x['latency_sec'] for x in cloud_recs.values()]):.2f}s")
else:
    cloud_recs = load_done(CLOUD_OUT)
    print("RUN_CLOUD=false -> using cached:", len(cloud_recs))

cloud: 280 cached, 0 to generate
cloud done: 280 rows, 0 API errors, avg prompt tokens 29623, avg latency 2.20s


## 5. Local 생성: Qwen2.5-Coder-7B-Instruct (DirectTransfer)

`local_models/qwen25_coder_7b`를 bf16으로 로드해 같은 메시지로 greedy 생성합니다
(chat template 적용, max_new_tokens 제한). GPU 단일 스트림 기준의 행별 지연과 입력 토큰 수를
기록하며, `local_generations.jsonl`로 재개 가능합니다.

In [5]:
# ============================================================
# Local generation with Qwen2.5-Coder-7B-Instruct (resume-capable)
# ============================================================
LOCAL_OUT = ART_DIR / "local_generations.jsonl"
if RUN_LOCAL:
    import torch
    from transformers import AutoModelForCausalLM
    done = load_done(LOCAL_OUT)
    todo = [i for i in range(min(N_ROWS, len(rows280))) if i not in done]
    print(f"local: {len(done)} cached, {len(todo)} to generate")
    if todo:
        model = AutoModelForCausalLM.from_pretrained(LOCAL_MODEL_DIR, dtype=torch.bfloat16,
                                                     device_map="cuda:0")
        model.eval()
        with LOCAL_OUT.open("a", encoding="utf-8") as f:
            for n, i in enumerate(todo, 1):
                r = rows280[i]
                msgs = [{"role": "system", "content": build_system_prompt(r.get("connected_devices", ""))},
                        {"role": "user", "content": user_prompt(r["command_eng"])}]
                enc = qwen_tok.apply_chat_template(msgs, add_generation_prompt=True,
                                                   return_tensors="pt", return_dict=True).to(model.device)
                n_in = int(enc["input_ids"].shape[1])
                t0 = time.perf_counter()
                with torch.inference_mode():
                    out_ids = model.generate(**enc, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                                             pad_token_id=qwen_tok.eos_token_id)
                latency = time.perf_counter() - t0
                content = qwen_tok.decode(out_ids[0][n_in:], skip_special_tokens=True)
                rec = {"row": i, "category": r["category"], "index": r["index"],
                       "command_eng": r["command_eng"], "content": content,
                       "parsed": parse_joilang_json(content),
                       "usage": {"prompt_tokens": n_in,
                                 "completion_tokens": int(out_ids.shape[1] - n_in)},
                       "latency_sec": round(latency, 3), "error": ""}
                f.write(json.dumps(rec, ensure_ascii=False) + "\n"); f.flush()
                if n % 10 == 0 or n == len(todo):
                    print(f"  local {n}/{len(todo)} (row {i + 1}) {latency:.1f}s", flush=True)
        del model
        torch.cuda.empty_cache()
    local_recs = load_done(LOCAL_OUT)
    print(f"local done: {len(local_recs)} rows, "
          f"avg prompt tokens {np.mean([x['usage'].get('prompt_tokens', 0) for x in local_recs.values()]):.0f}, "
          f"avg latency {np.mean([x['latency_sec'] for x in local_recs.values()]):.2f}s")
else:
    local_recs = load_done(LOCAL_OUT)
    print("RUN_LOCAL=false -> using cached:", len(local_recs))

local: 280 cached, 0 to generate
local done: 280 rows, avg prompt tokens 30107, avg latency 8.52s


## 6. 평가: utils/evaluation (IR/z3) + strict DET

두 축으로 평가합니다.

- `utils/evaluation/compare_soplang_ir.compare_codes`: SOPLang 파서로 gt/생성 코드를 AST로 만들고
  z3로 조건 논리 동치를 검사, `script_similarity`(문자열)·`ast_similarity`(정규화 AST)를 계산합니다.
  파서가 못 읽는 출력은 실패로 기록합니다. 사전 게이트로 **gt-vs-gt 자기평가**(파서 적합성)도 보고합니다.
- strict DET(`utils/det_evaluator.strict_det_evaluate_row`): 서버 공식 지표. `det_pass@70` 비율을
  논문의 DETPass와, `ast_similarity` 평균을 S_DET와 대응시켜 비교합니다.

In [6]:
# ============================================================
# Evaluation with utils/evaluation + strict DET
# ============================================================
import importlib.util
import pandas as pd

assert str(V15_DIR) not in sys.path
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))
from utils.evaluation.compare_soplang_ir import compare_codes

spec = importlib.util.spec_from_file_location("det_evaluator", BASE_DIR / "utils" / "det_evaluator.py")
det_evaluator = importlib.util.module_from_spec(spec)
spec.loader.exec_module(det_evaluator)
strict_det_evaluate_row = det_evaluator.strict_det_evaluate_row

def norm_gold(gt_str):
    g = json.loads(gt_str)
    return {"cron": str(g.get("cron", "")), "period": g.get("period", -1),
            "script": str(g.get("script", ""))}

def norm_pred(parsed):
    if not isinstance(parsed, dict):
        return None
    period = parsed.get("period", -1)
    try:
        period = int(period)
    except Exception:
        pass
    return {"cron": str(parsed.get("cron", "")), "period": period,
            "script": str(parsed.get("code", parsed.get("script", "")))}

def safe_compare(gold, pred):
    try:
        return compare_codes(gold, pred), ""
    except Exception as exc:
        return None, str(exc)[:120]

# --- gt-vs-gt parser sanity (utils/evaluation parser on the 280 gt idioms) ---
gt_ok = gt_fail = 0
for r in rows280[:N_ROWS]:
    g = norm_gold(r["gt"])
    res, err = safe_compare(g, g)
    if res and res["script_similarity"] >= 0.999:
        gt_ok += 1
    else:
        gt_fail += 1
print(f"[sanity] compare_codes gt-vs-gt: {gt_ok}/{gt_ok + gt_fail} self-consistent")

def evaluate_model(recs, label, normalize_period=False):
    out = []
    for i in sorted(recs):
        rec = recs[i]
        row = rows280[i]
        gold = norm_gold(row["gt"])
        pred = norm_pred(parse_joilang_json(rec.get("content", "")))
        if normalize_period and pred is not None:
            # the v0_6/v13 prompt lineage marks one-shot scenarios with period -1
            # while the ver2.0.1 gt uses period 0; both mean "no repetition"
            if pred["period"] in (-1, 0) and gold["period"] in (-1, 0):
                pred["period"] = gold["period"]
        entry = {"row": i, "category": row["category"], "label": label,
                 "parse_ok": pred is not None,
                 "prompt_tokens": rec.get("usage", {}).get("prompt_tokens", 0),
                 "latency_sec": rec.get("latency_sec", 0.0)}
        if pred is None:
            entry.update({"logic_equivalent": False, "script_similarity": 0.0,
                          "ast_similarity": 0.0, "cron_equal": False, "period_equal": False,
                          "ir_error": "unparseable_output"})
        else:
            res, err = safe_compare(gold, pred)
            if res is None:
                entry.update({"logic_equivalent": False, "script_similarity": 0.0,
                              "ast_similarity": 0.0, "cron_equal": False, "period_equal": False,
                              "ir_error": err})
            else:
                entry.update({k: res[k] for k in ("logic_equivalent", "script_similarity",
                                                  "ast_similarity", "cron_equal", "period_equal")})
                entry["ir_error"] = ""
        candidate = dict(pred, code=pred["script"]) if pred is not None else str(rec.get("content", ""))
        ev = strict_det_evaluate_row(row=row, candidate=candidate,
                                     row_no=f"{row['category']}-{row['index']}",
                                     det_threshold=DET_THRESHOLD)
        entry.update({"det_score": float(ev["det_score"]), "det_pass": bool(ev["det_pass"]),
                      "gt_exact": bool(ev["gt_exact"]),
                      "failure_reasons": ",".join(ev["failure_reasons"])})
        out.append(entry)
    return pd.DataFrame(out)

def summarize(df):
    return df.groupby("label").agg(
        n=("row", "count"),
        det_pass_pct=("det_pass", lambda s: 100 * s.mean()),
        det_score_mean=("det_score", "mean"),
        logic_equiv_pct=("logic_equivalent", lambda s: 100 * s.mean()),
        ast_sim_mean=("ast_similarity", "mean"),
        script_sim_mean=("script_similarity", "mean"),
        cron_eq_pct=("cron_equal", lambda s: 100 * s.mean()),
        period_eq_pct=("period_equal", lambda s: 100 * s.mean()),
        prompt_tokens=("prompt_tokens", "mean"),
        latency_sec=("latency_sec", "mean"),
    ).round(2)

MODELS = []
if cloud_recs:
    MODELS.append((cloud_recs, f"{CLOUD_MODEL} (CloudRef)"))
if local_recs:
    MODELS.append((local_recs, "Qwen2.5-Coder-7B (DirectTransfer)"))

df = pd.concat([evaluate_model(r, l) for r, l in MODELS], ignore_index=True)
df.to_csv(ART_DIR / "evaluation_rows.csv", index=False)
print("\n=== measured, RAW (period 규약 그대로) ===")
print(summarize(df).to_string())

df_norm = pd.concat([evaluate_model(r, l, normalize_period=True) for r, l in MODELS], ignore_index=True)
df_norm.to_csv(ART_DIR / "evaluation_rows_period_normalized.csv", index=False)
print("\n=== measured, period-convention NORMALIZED (-1 == 0 for one-shot) ===")
print(summarize(df_norm).to_string())

print("\n--- per-category det_pass % (normalized) ---")
piv = df_norm.pivot_table(index="category", columns="label", values="det_pass", aggfunc="mean") * 100
print(piv.round(1).to_string())

import collections as _c
for label, g in df_norm.groupby("label"):
    fr = _c.Counter(x for s in g[~g.det_pass]["failure_reasons"].fillna("") for x in str(s).split(",") if x)
    print(f"\n{label} top failure reasons: {fr.most_common(6)}")

paper = pd.DataFrame([
    {"label": "paper: GPT-4.1-mini CloudRef", "DETPass_pct": 94.5, "S_DET": 0.89,
     "input_tokens": 3500, "latency_sec": 2.85},
    {"label": "paper: Qwen2.5-Coder-7B DirectTransfer", "DETPass_pct": 87.8, "S_DET": 0.73,
     "input_tokens": 3500, "latency_sec": 2.28},
]).set_index("label")
print("\n=== paper targets (Table 5; ReplayFit 제외) ===")
print(paper.to_string())
summarize(df).to_csv(ART_DIR / "evaluation_summary_raw.csv")
summarize(df_norm).to_csv(ART_DIR / "evaluation_summary_period_normalized.csv")
print("\nsaved ->", ART_DIR / "evaluation_rows.csv", "|", ART_DIR / "evaluation_summary_period_normalized.csv")

[sanity] compare_codes gt-vs-gt: 280/280 self-consistent



=== measured, RAW (period 규약 그대로) ===
                                     n  det_pass_pct  det_score_mean  logic_equiv_pct  ast_sim_mean  script_sim_mean  cron_eq_pct  period_eq_pct  prompt_tokens  latency_sec
label                                                                                                                                                                       
Qwen2.5-Coder-7B (DirectTransfer)  280         26.79           59.75            92.14          0.69             0.67        95.00          16.79       30106.56         8.52
gpt-4.1-mini (CloudRef)            280         63.21           73.60            92.50          0.73             0.72        93.93          21.43       29622.99         2.20



=== measured, period-convention NORMALIZED (-1 == 0 for one-shot) ===
                                     n  det_pass_pct  det_score_mean  logic_equiv_pct  ast_sim_mean  script_sim_mean  cron_eq_pct  period_eq_pct  prompt_tokens  latency_sec
label                                                                                                                                                                       
Qwen2.5-Coder-7B (DirectTransfer)  280         34.29           65.08            92.14          0.69             0.67        95.00          87.86       30106.56         8.52
gpt-4.1-mini (CloudRef)            280         69.29           78.91            92.50          0.73             0.72        93.93          92.14       29622.99         2.20

--- per-category det_pass % (normalized) ---
label     Qwen2.5-Coder-7B (DirectTransfer)  gpt-4.1-mini (CloudRef)
category                                                            
1                                      43.3          

## 7. Summary

- 게이트: premapping 장치 recall 1.0 (retrieval 140/140) 확인 후 진행.
- 조건: v0_6 full prompt + ver2.0.1 service list(호출명 규약 변환), temperature 0,
  동일 메시지로 cloud/local 생성.
- 비교 기준: `det_pass@70` ↔ 논문 DETPass, `ast_similarity` 평균 ↔ S_DET,
  실측 prompt tokens/지연 ↔ 논문 토큰/지연 열. 논문 수치는 사전 측정 전 값이므로
  "cloud > local의 서열과 대략적 수준"이 재현되는지를 봅니다.
- 산출물: `artifacts/fullprompt_runs/<RUN_TAG>/`
  (merged prompt, cloud/local generations jsonl, evaluation_rows/summary csv)

In [7]:
# ============================================================
# Artifact listing
# ============================================================
for p in sorted(ART_DIR.rglob("*")):
    if p.is_file():
        print(f"{p.stat().st_size:>12,d}  {p.relative_to(ART_DIR)}")

     208,930  cloud_generations.jsonl
      94,063  evaluation_rows.csv
      87,305  evaluation_rows_period_normalized.csv
         305  evaluation_summary.csv
         304  evaluation_summary_period_normalized.csv
         303  evaluation_summary_raw.csv
     198,425  local_generations.jsonl
     118,409  merged_system_prompt_ver201.md
